# ДЗ 6. CNN для классификации изображений: архитектура и обучение

Продолжение `Imagenette.ipynb` с лекции — тот же паттерн (Dataset → transforms → DataLoader → CNN → цикл обучения), но на новом датасете и с более глубоким разбором того, что реально влияет на качество: аугментации, BatchNorm, расписание learning rate, интерпретируемость.

**Датасет.** Рекомендуем **EuroSAT** (`torchvision.datasets.EuroSAT`) — снимки со спутника Sentinel-2, 10 классов землепользования, 64×64, ~27000 изображений. Код ниже написан под интерфейс ImageFolder-подобных датасетов и переносится на STL10 с минимальными изменениями.

**Бюджет.** Полный датасет EuroSAT + `SimpleCNN` — разумный объём для GPU Colab. При использовании CPU допустимо уменьшить train-подвыборку.


In [ ]:
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader, Subset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random, time, math, gc
from IPython.display import display

torch.manual_seed(42)
random.seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_WORKERS = 2 if device.type == 'cuda' else 0
BATCH_SIZE = 128 if device.type == 'cuda' else 64
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

transform_plain = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])
transform_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])
full_dataset = torchvision.datasets.EuroSAT(root='/content/case3_data', transform=None, download=True)
classes = full_dataset.classes
labels = np.array(full_dataset.targets)

rng = np.random.default_rng(42)
train_idx, val_idx, test_idx = [], [], []
for c in np.unique(labels):
    idx = np.where(labels == c)[0]
    rng.shuffle(idx)
    n = len(idx)
    a, b = int(0.70*n), int(0.85*n)
    train_idx.extend(idx[:a]); val_idx.extend(idx[a:b]); test_idx.extend(idx[b:])
train_idx, val_idx, test_idx = map(np.array, (train_idx, val_idx, test_idx))
print(classes)
print(f'train={len(train_idx)} val={len(val_idx)} test={len(test_idx)}')

plain_dataset = torchvision.datasets.EuroSAT(root='/content/case3_data', transform=transform_plain, download=False)
aug_dataset = torchvision.datasets.EuroSAT(root='/content/case3_data', transform=transform_aug, download=False)
train_ds = Subset(plain_dataset, train_idx)
train_aug_ds = Subset(aug_dataset, train_idx)
val_ds = Subset(plain_dataset, val_idx)
test_ds = Subset(plain_dataset, test_idx)

def loader(ds, shuffle=False, seed=42):
    g = torch.Generator().manual_seed(seed)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=device.type == 'cuda', generator=g)

train_loader = loader(train_ds, True)
val_loader = loader(val_ds)
test_loader = loader(test_ds)

criterion = nn.CrossEntropyLoss()

def train_epoch(model, dl, optimizer, track_loss=False):
    model.train(); total_loss = total_correct = total = 0; batch_losses = []
    use_amp = device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    for x, y in dl:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        if use_amp:
            with torch.autocast('cuda', dtype=torch.float16):
                out = model(x); loss = criterion(out, y)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        else:
            out = model(x); loss = criterion(out, y); loss.backward(); optimizer.step()
        bs = y.size(0); total += bs; total_loss += loss.item() * bs
        total_correct += (out.argmax(1) == y).sum().item()
        if track_loss: batch_losses.append(loss.item())
    return total_loss/total, total_correct/total, batch_losses

@torch.no_grad()
def evaluate(model, dl):
    model.eval(); total_loss = total_correct = total = 0
    for x, y in dl:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        out = model(x); loss = criterion(out, y); bs = y.size(0)
        total += bs; total_loss += loss.item()*bs; total_correct += (out.argmax(1) == y).sum().item()
    return total_loss/total, total_correct/total

def fit(model, train_dl, epochs=10, lr=1e-3, scheduler=None, track_loss=False):
    model.to(device); opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'lr': [], 'iter_loss': [], 'epoch_time': []}
    for ep in range(epochs):
        t0 = time.perf_counter()
        tr_loss, tr_acc, it_loss = train_epoch(model, train_dl, opt, track_loss)
        va_loss, va_acc = evaluate(model, val_loader)
        history['train_loss'].append(tr_loss); history['train_acc'].append(tr_acc)
        history['val_loss'].append(va_loss); history['val_acc'].append(va_acc)
        history['lr'].append(scheduler.get_last_lr()[0] if scheduler is not None else opt.param_groups[0]['lr'])
        history['epoch_time'].append(time.perf_counter()-t0)
        if track_loss: history['iter_loss'].extend(it_loss)
        if scheduler is not None: scheduler.step()
        print(f'{ep+1:02d}/{epochs}: train={tr_acc:.4f} val={va_acc:.4f}')
    return model, history

def test_acc(model):
    return evaluate(model, test_loader)[1]

class SimpleCNN(nn.Module):
    def __init__(self, use_bn=False, channels=(32,64,128)):
        super().__init__()
        blocks=[]; in_ch=3
        for ch in channels:
            layers=[nn.Conv2d(in_ch,ch,3,padding=1)]
            if use_bn: layers.append(nn.BatchNorm2d(ch))
            layers += [nn.ReLU(inplace=True), nn.MaxPool2d(2)]
            blocks.append(nn.Sequential(*layers)); in_ch=ch
        self.features=nn.ModuleList(blocks)
        self.classifier=nn.Linear(channels[-1]*8*8, len(classes))
    def forward(self,x):
        for block in self.features: x=block(x)
        return self.classifier(x.flatten(1))

def cleanup(*objs):
    for obj in objs:
        del obj
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


## Уровень 1. Базовый

### Задача 1.1. `SimpleCNN` с лекции на новом датасете

Переносим архитектуру из лекционного ноутбука на EuroSAT.

In [ ]:
# 1.1 SimpleCNN
model_11, history_11 = fit(SimpleCNN(), train_loader, epochs=10, lr=1e-3)
print(f'Test accuracy: {test_acc(model_11):.4f}')

plt.figure(figsize=(8,4))
plt.plot(history_11['train_acc'], label='train')
plt.plot(history_11['val_acc'], label='val')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.title('SimpleCNN')
plt.legend(); plt.grid(alpha=.2); plt.show()

x, y = next(iter(test_loader)); x = x.to(device)
with torch.no_grad(): pred = model_11(x).argmax(1).cpu()
mean = torch.tensor((0.485,0.456,0.406)).view(3,1,1)
std = torch.tensor((0.229,0.224,0.225)).view(3,1,1)
fig, ax = plt.subplots(2,4,figsize=(12,6))
for a, img, yt, yp in zip(ax.flat, x.cpu()[:8], y[:8], pred[:8]):
    img = (img*std+mean).clamp(0,1)
    a.imshow(img.permute(1,2,0)); a.set_title(f'{classes[yt]} → {classes[yp]}'); a.axis('off')
plt.tight_layout(); plt.show()


### Задача 1.2. Влияние аугментаций на переобучение

Используем небольшой train subset и сравниваем одинаковую архитектуру без аугментаций и с ними.

In [ ]:
# 1.2 Без augmentation и с augmentation
small_idx = rng.choice(train_idx, 500, replace=False)
plain_small = loader(Subset(plain_dataset, small_idx), True, seed=42)
aug_small = loader(Subset(aug_dataset, small_idx), True, seed=42)

_, h_plain = fit(SimpleCNN(), plain_small, epochs=20, lr=1e-3)
_, h_aug = fit(SimpleCNN(), aug_small, epochs=20, lr=1e-3)

gap_plain = h_plain['train_acc'][-1] - h_plain['val_acc'][-1]
gap_aug = h_aug['train_acc'][-1] - h_aug['val_acc'][-1]
aug_results = pd.DataFrame({
    'model':['without augmentation','with augmentation'],
    'train_acc':[h_plain['train_acc'][-1], h_aug['train_acc'][-1]],
    'val_acc':[h_plain['val_acc'][-1], h_aug['val_acc'][-1]],
    'gap':[gap_plain, gap_aug]
})
display(aug_results)
plt.figure(figsize=(9,5))
plt.plot(h_plain['train_acc'], label='plain train'); plt.plot(h_plain['val_acc'], label='plain val')
plt.plot(h_aug['train_acc'], '--', label='aug train'); plt.plot(h_aug['val_acc'], '--', label='aug val')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.title('Augmentation on a small train subset')
plt.legend(); plt.grid(alpha=.2); plt.show()
print(f'Фактическое изменение val accuracy: {h_aug["val_acc"][-1]-h_plain["val_acc"][-1]:+.4f}')
print(f'Фактическое изменение gap: {gap_aug-gap_plain:+.4f}')


### Задача 1.3. Визуализация фильтров и feature maps

In [ ]:
# 1.3 Фильтры и feature maps
def show_filters(model, title):
    w = model.features[0][0].weight.detach().cpu()
    n = len(w); cols=8; rows=math.ceil(n/cols)
    fig, ax = plt.subplots(rows, cols, figsize=(12, 1.7*rows)); ax=np.array(ax).reshape(-1)
    for i in range(n):
        f=w[i].permute(1,2,0); f=(f-f.min())/(f.max()-f.min()+1e-8)
        ax[i].imshow(f); ax[i].axis('off')
    for a in ax[n:]: a.axis('off')
    fig.suptitle(title); plt.tight_layout(); plt.show()

def get_maps(model, img):
    model.eval().to(device); x=img.unsqueeze(0).to(device); maps=[]
    with torch.no_grad():
        for block in model.features:
            x=block(x); maps.append(x[0].cpu())
    return maps

untrained = SimpleCNN().to(device)
show_filters(untrained, 'Untrained filters')
show_filters(model_11, 'Trained filters')
img, _ = test_ds[0]
maps_u, maps_t = get_maps(untrained, img), get_maps(model_11, img)
fig, ax = plt.subplots(3,3,figsize=(10,10))
for i in range(3):
    ax[i,0].imshow(maps_u[i][0], cmap='viridis'); ax[i,0].set_title(f'Untrained block {i+1}')
    ax[i,1].imshow(maps_t[i][0], cmap='viridis'); ax[i,1].set_title(f'Trained block {i+1}, ch 1')
    ax[i,2].imshow(maps_t[i][min(1, maps_t[i].shape[0]-1)], cmap='viridis'); ax[i,2].set_title(f'Trained block {i+1}, ch 2')
for a in ax.flat: a.axis('off')
plt.tight_layout(); plt.show()


## Уровень 2. Средний

### Задача 2.1. Глубина и ширина архитектуры

In [ ]:
# 2.1 Сравнение глубины и ширины
class ConfigurableCNN(nn.Module):
    def __init__(self, n_blocks=3, base_channels=16):
        super().__init__()
        assert 1 <= n_blocks <= 5
        blocks=[]; in_ch=3; chs=[]
        for i in range(n_blocks):
            ch=base_channels*(2**i); chs.append(ch)
            blocks.append(nn.Sequential(nn.Conv2d(in_ch,ch,3,padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2)))
            in_ch=ch
        self.features=nn.ModuleList(blocks)
        self.classifier=nn.Linear(chs[-1]*(64//(2**n_blocks))**2, len(classes))
    def forward(self,x):
        for block in self.features: x=block(x)
        return self.classifier(x.flatten(1))

configs=[('shallow_narrow',2,16),('shallow_wide',2,32),('deep_narrow',4,16),('deep_wide',4,32)]
rows=[]
for name, depth, width in configs:
    print(f'
[2.1 {len(rows)+1}/{len(configs)}] {name}')
    torch.manual_seed(42); np.random.seed(42); random.seed(42)
    m=ConfigurableCNN(depth,width)
    params=sum(p.numel() for p in m.parameters())
    t=time.perf_counter(); m,h=fit(m,train_loader,epochs=8,lr=1e-3); elapsed=time.perf_counter()-t
    rows.append({'model':name,'params':params,'test_accuracy':test_acc(m),'epoch_time':elapsed/8})
    cleanup(m)
results_21=pd.DataFrame(rows)
display(results_21)
fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].scatter(results_21['params'],results_21['test_accuracy'])
for _,r in results_21.iterrows(): ax[0].annotate(r['model'],(r['params'],r['test_accuracy']))
ax[0].set(xlabel='Parameters',ylabel='Test accuracy',title='Accuracy vs parameters')
ax[1].scatter(results_21['epoch_time'],results_21['test_accuracy'])
for _,r in results_21.iterrows(): ax[1].annotate(r['model'],(r['epoch_time'],r['test_accuracy']))
ax[1].set(xlabel='Seconds / epoch',ylabel='Test accuracy',title='Accuracy vs epoch time')
plt.tight_layout(); plt.show()
print('Вывод по данным:', results_21.loc[results_21['test_accuracy'].idxmax(),'model'], 'имеет наибольшую test accuracy в этом запуске.')


### Задача 2.2. Batch Normalization: устойчивость к learning rate

In [ ]:
# 2.2 BatchNorm × learning rate
rows=[]; histories={}
for use_bn, lr in [(False,0.001),(True,0.001),(False,0.02),(True,0.02)]:
    name=f'BN={use_bn}, lr={lr}'
    print(f'
[2.2 {len(rows)+1}/4] {name}')
    torch.manual_seed(42); np.random.seed(42); random.seed(42)
    m,h=fit(SimpleCNN(use_bn=use_bn), train_loader, epochs=8, lr=lr, track_loss=True)
    histories[name]=h
    rows.append({'use_bn':use_bn,'lr':lr,'test_accuracy':test_acc(m),'best_val_accuracy':max(h['val_acc'])})
    cleanup(m)
bn_results=pd.DataFrame(rows); display(bn_results)
plt.figure(figsize=(11,5))
for name,h in histories.items(): plt.plot(h['iter_loss'], label=name, alpha=.8)
plt.xlabel('Training iteration'); plt.ylabel('Loss'); plt.title('BatchNorm and learning rate')
plt.legend(); plt.grid(alpha=.2); plt.show()


### Задача 2.3. Расписания learning rate

In [ ]:
# 2.3 Learning-rate schedules
rows=[]; histories={}
for name in ['constant','step','cosine']:
    print(f'
[2.3 {len(rows)+1}/3] {name}')
    torch.manual_seed(42); np.random.seed(42); random.seed(42)
    m=SimpleCNN().to(device); opt=torch.optim.Adam(m.parameters(),lr=1e-3)
    if name=='constant': scheduler=torch.optim.lr_scheduler.LambdaLR(opt,lambda _:1.0)
    elif name=='step': scheduler=torch.optim.lr_scheduler.StepLR(opt,step_size=5,gamma=0.1)
    else: scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=15)
    h={'val_acc':[],'lr':[]}
    for ep in range(15):
        train_epoch(m,train_loader,opt); h['val_acc'].append(evaluate(m,val_loader)[1])
        h['lr'].append(scheduler.get_last_lr()[0]); scheduler.step()
        print(f'{ep+1:02d}/15: val={h["val_acc"][-1]:.4f} lr={h["lr"][-1]:.6f}')
    histories[name]=h; rows.append({'schedule':name,'test_accuracy':test_acc(m),'best_val_accuracy':max(h['val_acc'])})
    cleanup(m)
schedule_results=pd.DataFrame(rows); display(schedule_results)
fig,ax=plt.subplots(1,2,figsize=(12,4))
for name,h in histories.items(): ax[0].plot(h['val_acc'],label=name); ax[1].plot(h['lr'],label=name)
ax[0].set(xlabel='Epoch',ylabel='Val accuracy',title='Validation accuracy')
ax[1].set(xlabel='Epoch',ylabel='Learning rate',title='Learning rate')
for a in ax: a.legend(); a.grid(alpha=.2)
plt.tight_layout(); plt.show()
print('Вывод по данным:', schedule_results.loc[schedule_results['test_accuracy'].idxmax(),'schedule'], 'дал наибольшую test accuracy в этом запуске.')


---
Численные результаты и выводы должны формироваться по фактическому запуску ноутбука.